# 00 - Data Preprocessing

Notebook khusus preprocessing data untuk pipeline tesis IHSG. Output-nya adalah `df_merged` joblib yang dipakai oleh:
- `hyperparameter_tuning.ipynb` (Optuna)
- `phase1_screening.ipynb` (eksperimen)
- notebook phase berikutnya

Tidak ada training model, grid search, atau cross-validation di sini - itu semua di notebook downstream.

## Langkah
1. Load semua CSV dari GitHub
2. Merge target + macro + commodity + regional
3. ADF test sebelum & sesudah transformasi (untuk dokumentasi tesis)
4. Save `df_merged` ke joblib


## 1. Imports & Konfigurasi

In [1]:
import pandas as pd
import numpy as np
import joblib
import os
from datetime import datetime
from statsmodels.tsa.stattools import adfuller
import warnings

warnings.filterwarnings("ignore")

URLS = {
    "ihsg":      "https://raw.githubusercontent.com/ravsssh/tesis-hakam/refs/heads/main/dataset/ihsg.csv",
    "bi_rate":   "https://raw.githubusercontent.com/ravsssh/tesis-hakam/refs/heads/main/dataset/bi_interest_rate.csv",
    "cpi":       "https://raw.githubusercontent.com/ravsssh/tesis-hakam/refs/heads/main/dataset/cpi.csv",
    "m2":        "https://raw.githubusercontent.com/ravsssh/tesis-hakam/refs/heads/main/dataset/m2.csv",
    "npl_ratio": "https://raw.githubusercontent.com/ravsssh/tesis-hakam/refs/heads/main/dataset/npl_ratio.csv",
    "usdidr":    "https://raw.githubusercontent.com/ravsssh/tesis-hakam/refs/heads/main/dataset/usd_idr.csv",
    "coal":      "https://raw.githubusercontent.com/ravsssh/tesis-hakam/refs/heads/main/dataset/Coal.csv",
    "copper":    "https://raw.githubusercontent.com/ravsssh/tesis-hakam/refs/heads/main/dataset/Copper.csv",
    "nickel":    "https://raw.githubusercontent.com/ravsssh/tesis-hakam/refs/heads/main/dataset/Nickel.csv",
    "silver":    "https://raw.githubusercontent.com/ravsssh/tesis-hakam/refs/heads/main/dataset/Silver.csv",
    "tin":       "https://raw.githubusercontent.com/ravsssh/tesis-hakam/refs/heads/main/dataset/Tin.csv",
    "sti":       "https://raw.githubusercontent.com/ravsssh/tesis-hakam/refs/heads/main/dataset/STI.csv",
    "gold":      "https://raw.githubusercontent.com/ravsssh/tesis-hakam/refs/heads/main/dataset/Gold.csv",
    "wti":       "https://raw.githubusercontent.com/ravsssh/tesis-hakam/refs/heads/main/dataset/wti.csv",
    "ust10y":    "https://raw.githubusercontent.com/ravsssh/tesis-hakam/refs/heads/main/dataset/ustressury10y.csv",
    "gdp":       "https://raw.githubusercontent.com/ravsssh/tesis-hakam/refs/heads/main/dataset/gdp.csv",
}

# Transformasi rule
LEVEL_VARS = ["M2", "USDIDR", "Coal", "Copper", "Nickel", "Silver", "Tin", "STI", "Gold", "WTI", "GDP"]
RATE_VARS  = ["BI_Rate", "CPI", "NPL_Ratio", "US_Treasury_10Y"]

DATE_START = "2015-01-01"
DATE_END   = "2025-01-31"

print(f"Period: {DATE_START} to {DATE_END}")
print(f"LEVEL_VARS ({len(LEVEL_VARS)}): {LEVEL_VARS}")
print(f"RATE_VARS  ({len(RATE_VARS)}): {RATE_VARS}")

Period: 2015-01-01 to 2025-01-31
LEVEL_VARS (11): ['M2', 'USDIDR', 'Coal', 'Copper', 'Nickel', 'Silver', 'Tin', 'STI', 'Gold', 'WTI', 'GDP']
RATE_VARS  (4): ['BI_Rate', 'CPI', 'NPL_Ratio', 'US_Treasury_10Y']


## 2. Load Target (IHSG)

In [2]:
df_ihsg = pd.read_csv(URLS["ihsg"])
df_ihsg.columns = ["date", "IHSG"]
df_ihsg["date"] = pd.to_datetime(df_ihsg["date"], format="%d/%m/%Y")
df_ihsg = df_ihsg.sort_values("date").reset_index(drop=True)
print(f"IHSG: {len(df_ihsg)} rows | {df_ihsg['date'].min().date()} to {df_ihsg['date'].max().date()}")


IHSG: 2660 rows | 2015-01-02 to 2025-12-30


## 3. Load Macro Variables (existing)

In [3]:
# BI Rate
df_bi = pd.read_csv(URLS["bi_rate"])
df_bi.columns = ["date", "BI_Rate"]
df_bi["date"] = pd.to_datetime(df_bi["date"], format="%d/%m/%y")
df_bi["BI_Rate"] = df_bi["BI_Rate"].str.replace("%", "").astype(float)
df_bi = df_bi.sort_values("date").reset_index(drop=True)

# CPI
df_cpi = pd.read_csv(URLS["cpi"])
df_cpi.columns = ["date", "CPI"]
df_cpi["date"] = pd.to_datetime(df_cpi["date"], format="%Y-%m-%d")
df_cpi = df_cpi.sort_values("date").reset_index(drop=True)

# M2
df_m2 = pd.read_csv(URLS["m2"])
df_m2.columns = ["date", "M2"]
df_m2["date"] = pd.to_datetime(df_m2["date"], format="%m/%d/%Y")
df_m2["M2"] = df_m2["M2"].str.replace('"', "").str.replace(",", "").astype(float)
df_m2 = df_m2.sort_values("date").reset_index(drop=True)

# NPL Ratio
df_npl = pd.read_csv(URLS["npl_ratio"])
df_npl.columns = ["date", "NPL_Ratio"]
df_npl["date"] = pd.to_datetime(df_npl["date"], format="%d/%m/%y")
df_npl["NPL_Ratio"] = df_npl["NPL_Ratio"].str.replace("%", "").astype(float)
df_npl = df_npl.sort_values("date").reset_index(drop=True)

# USD/IDR
df_usd = pd.read_csv(URLS["usdidr"])
df_usd.columns = ["date", "USDIDR"]
df_usd["date"] = pd.to_datetime(df_usd["date"], format="%d/%m/%Y")
df_usd["USDIDR"] = (
    df_usd["USDIDR"].astype(str)
    .str.replace('"', "").str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False).astype(float)
)
df_usd = df_usd.sort_values("date").reset_index(drop=True)

for name, df in [("BI_Rate",df_bi),("CPI",df_cpi),("M2",df_m2),("NPL_Ratio",df_npl),("USDIDR",df_usd)]:
    print(f"{name:10s}: {len(df):5d} rows | {df['date'].min().date()} to {df['date'].max().date()}")


BI_Rate   :   121 rows | 2015-01-01 to 2025-01-01
CPI       :   697 rows | 1968-01-31 to 2026-01-31
M2        :   121 rows | 2015-01-01 to 2025-01-01
NPL_Ratio :   121 rows | 2015-01-01 to 2025-01-01
USDIDR    :  2854 rows | 2015-01-01 to 2026-02-23


## 4. Load Macro Variables (NEW: GDP, US_Treasury_10Y, WTI)

In [4]:
# WTI Crude Oil - daily level, mm/dd/yyyy
df_wti = pd.read_csv(URLS["wti"])
df_wti.columns = ["date", "WTI"]
df_wti["date"] = pd.to_datetime(df_wti["date"], format="%m/%d/%Y")
df_wti["WTI"] = df_wti["WTI"].astype(float)
df_wti = df_wti.sort_values("date").reset_index(drop=True)

# US Treasury 10Y - daily rate, mm/dd/yyyy
df_ust = pd.read_csv(URLS["ust10y"])
df_ust.columns = ["date", "US_Treasury_10Y"]
df_ust["date"] = pd.to_datetime(df_ust["date"], format="%m/%d/%Y")
df_ust["US_Treasury_10Y"] = df_ust["US_Treasury_10Y"].astype(float)
df_ust = df_ust.sort_values("date").reset_index(drop=True)

# GDP - quarterly level, mm/dd/yyyy
df_gdp = pd.read_csv(URLS["gdp"])
df_gdp.columns = ["date", "GDP"]
df_gdp["date"] = pd.to_datetime(df_gdp["date"], format="%m/%d/%Y")
df_gdp["GDP"] = df_gdp["GDP"].astype(float)
df_gdp = df_gdp.sort_values("date").reset_index(drop=True)

for name, df in [("WTI",df_wti),("US_Treasury_10Y",df_ust),("GDP",df_gdp)]:
    print(f"{name:16s}: {len(df):5d} rows | {df['date'].min().date()} to {df['date'].max().date()}")


WTI             :  3224 rows | 2014-01-01 to 2026-04-08
US_Treasury_10Y :  3196 rows | 2014-01-09 to 2026-04-08
GDP             :   104 rows | 2000-01-01 to 2025-10-01


## 5. Load Commodities + Regional (STI)

In [5]:
def load_daily_csv(url, col_name):
    df = pd.read_csv(url)
    df.columns = ["date", col_name]
    parsed = False
    for fmt in ["%d/%m/%Y", "%m/%d/%Y", "%Y-%m-%d", "%d/%m/%y"]:
        try:
            df["date"] = pd.to_datetime(df["date"], format=fmt)
            parsed = True
            break
        except (ValueError, TypeError):
            continue
    if not parsed:
        df["date"] = pd.to_datetime(df["date"], dayfirst=True)
    if df[col_name].dtype == object:
        df[col_name] = (
            df[col_name].astype(str)
            .str.replace('"', "", regex=False)
            .str.replace(",", "", regex=False)
            .astype(float)
        )
    else:
        df[col_name] = df[col_name].astype(float)
    return df.sort_values("date").reset_index(drop=True)


daily_vars = {
    "Coal":   URLS["coal"],
    "Copper": URLS["copper"],
    "Nickel": URLS["nickel"],
    "Silver": URLS["silver"],
    "Tin":    URLS["tin"],
    "Gold":   URLS["gold"],
    "STI":    URLS["sti"],
}
daily_dfs = {name: load_daily_csv(url, name) for name, url in daily_vars.items()}

for name, df in daily_dfs.items():
    print(f"{name:8s}: {len(df):5d} rows | {df['date'].min().date()} to {df['date'].max().date()}")


Coal    :  2839 rows | 2015-01-02 to 2025-12-31
Copper  :  2780 rows | 2015-01-02 to 2025-12-31
Nickel  :  2774 rows | 2015-01-02 to 2025-12-31
Silver  :  2857 rows | 2015-01-01 to 2025-12-31
Tin     :  2780 rows | 2015-01-02 to 2025-12-31
Gold    :  2857 rows | 2015-01-01 to 2025-12-31
STI     :  2766 rows | 2015-01-02 to 2025-12-31


## 6. Merge Semua Variabel

Strategi merge:
- **Base:** IHSG daily (business days)
- **Monthly/Quarterly** (`BI_Rate, CPI, M2, NPL_Ratio, GDP`): `merge_asof` direction=`backward` -> value terakhir yang sudah rilis
- **Daily** (`USDIDR, WTI, US_Treasury_10Y, Coal, ..., STI`): left merge + `ffill` untuk isi weekend/holiday


In [6]:
df_merged = df_ihsg.copy()
df_merged = df_merged[
    (df_merged["date"] >= DATE_START) & (df_merged["date"] <= DATE_END)
].reset_index(drop=True)
print(f"Base IHSG ({DATE_START} to {DATE_END}): {len(df_merged)} rows")

# --- Monthly/Quarterly: merge_asof backward ---
monthly_dfs = {
    "BI_Rate":   df_bi,
    "CPI":       df_cpi,
    "M2":        df_m2,
    "NPL_Ratio": df_npl,
    "GDP":       df_gdp,
}
for var_name, df_m in monthly_dfs.items():
    df_merged = pd.merge_asof(
        df_merged.sort_values("date"),
        df_m[["date", var_name]].sort_values("date"),
        on="date", direction="backward"
    )
    print(f"  {var_name:16s} (asof backward): {df_merged[var_name].notna().sum()} non-null")

# --- Daily: left merge + ffill ---
daily_merge = {
    "USDIDR":          df_usd,
    "WTI":             df_wti,
    "US_Treasury_10Y": df_ust,
    **daily_dfs,
}
for var_name, df_d in daily_merge.items():
    df_merged = pd.merge(df_merged, df_d[["date", var_name]], on="date", how="left")
    df_merged[var_name] = df_merged[var_name].ffill()
    non_null = df_merged[var_name].notna().sum()
    null_ct  = df_merged[var_name].isna().sum()
    print(f"  {var_name:16s} (daily ffill):  {non_null} non-null, {null_ct} missing")

before = len(df_merged)
df_merged = df_merged.dropna().reset_index(drop=True)
print(f"\nDropped {before - len(df_merged)} rows with any NaN")
print(f"Final shape: {df_merged.shape}")
print(f"Date range: {df_merged['date'].min().date()} to {df_merged['date'].max().date()}")
print(f"Columns: {list(df_merged.columns)}")


Base IHSG (2015-01-01 to 2025-01-31): 2443 rows
  BI_Rate          (asof backward): 2443 non-null
  CPI              (asof backward): 2443 non-null
  M2               (asof backward): 2443 non-null
  NPL_Ratio        (asof backward): 2443 non-null
  GDP              (asof backward): 2443 non-null
  USDIDR           (daily ffill):  2443 non-null, 0 missing
  WTI              (daily ffill):  2443 non-null, 0 missing
  US_Treasury_10Y  (daily ffill):  2443 non-null, 0 missing
  Coal             (daily ffill):  2443 non-null, 0 missing
  Copper           (daily ffill):  2443 non-null, 0 missing
  Nickel           (daily ffill):  2443 non-null, 0 missing
  Silver           (daily ffill):  2443 non-null, 0 missing
  Tin              (daily ffill):  2443 non-null, 0 missing
  Gold             (daily ffill):  2443 non-null, 0 missing
  STI              (daily ffill):  2443 non-null, 0 missing

Dropped 0 rows with any NaN
Final shape: (2443, 17)
Date range: 2015-01-02 to 2025-01-31
Columns: ['d

## 7. ADF Test - Level (sebelum transformasi)

In [7]:
def adf_test(series, name):
    if isinstance(series, np.ndarray):
        series = series[~np.isnan(series)]
    else:
        series = series.dropna()
    result = adfuller(series, autolag="AIC")
    return {
        "variable": name,
        "ADF Statistic": result[0],
        "p-value": result[1],
        "Lags Used": result[2],
        "Observations Used": result[3],
        "Stationary": "Yes" if result[1] < 0.05 else "No",
    }


test_cols = [c for c in df_merged.columns if c != "date"]

print("=== ADF Test: LEVEL ===")
df_adf_level = pd.DataFrame([adf_test(df_merged[c], c) for c in test_cols])
display(df_adf_level)


=== ADF Test: LEVEL ===


,variable,ADF Statistic,p-value,Lags Used,Observations Used,Stationary
0,IHSG,-1.529615,0.518814,3,2439,No
1,BI_Rate,-1.890520,0.336534,22,2420,No
2,CPI,-0.738399,0.836551,21,2421,No
3,M2,0.627073,0.988264,21,2421,No
4,NPL_Ratio,-1.083827,0.721478,19,2423,No
5,GDP,0.150359,0.969336,0,2442,No
6,USDIDR,-1.961188,0.303846,4,2438,No
7,WTI,-1.952631,0.307721,12,2430,No
8,US_Treasury_10Y,-0.522333,0.887584,1,2441,No
9,Coal,-1.641815,0.461313,18,2424,No


## 8. ADF Test - Transformed

Aturan transformasi:
- `LEVEL_VARS` dan target `IHSG` -> `diff(log(x))` (log-return)
- `RATE_VARS` -> `diff(x)` (first difference)


In [8]:
print("=== ADF Test: TRANSFORMED ===")
transformed_results = []
for col in test_cols:
    s = df_merged[col].dropna()
    if col in LEVEL_VARS or col == "IHSG":
        transformed = np.diff(np.log(s))
        label = f"{col} (log-diff)"
    elif col in RATE_VARS:
        transformed = np.diff(s)
        label = f"{col} (diff)"
    else:
        transformed = s.values
        label = f"{col} (level)"
    transformed_results.append(adf_test(transformed, label))

df_adf_transformed = pd.DataFrame(transformed_results)
display(df_adf_transformed)


=== ADF Test: TRANSFORMED ===


,variable,ADF Statistic,p-value,Lags Used,Observations Used,Stationary
0,IHSG (log-diff),-27.637467,0.000000e+00,2,2439,Yes
1,BI_Rate (diff),-6.881872,1.426610e-09,21,2420,Yes
2,CPI (diff),-8.525295,1.073256e-13,20,2421,Yes
3,M2 (log-diff),-12.635400,1.473769e-23,21,2420,Yes
4,NPL_Ratio (diff),-14.304398,1.213725e-26,18,2423,Yes
5,GDP (log-diff),-49.671210,0.000000e+00,0,2441,Yes
6,USDIDR (log-diff),-21.168144,0.000000e+00,3,2438,Yes
7,WTI (log-diff),-9.784449,6.600444e-17,21,2420,Yes
8,US_Treasury_10Y (diff),-51.787630,0.000000e+00,0,2441,Yes
9,Coal (log-diff),-11.182985,2.474376e-20,17,2424,Yes


## 9. Save `df_merged` + ADF Results

In [9]:
save_dir = "saved_models"
os.makedirs(save_dir, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

merged_path = f"{save_dir}/df_merged_{timestamp}.joblib"
joblib.dump(df_merged, merged_path)

df_adf_level.to_csv(f"{save_dir}/adf_level_{timestamp}.csv", index=False)
df_adf_transformed.to_csv(f"{save_dir}/adf_transformed_{timestamp}.csv", index=False)

print("Saved:")
print(f"  {merged_path}")
print(f"  {save_dir}/adf_level_{timestamp}.csv")
print(f"  {save_dir}/adf_transformed_{timestamp}.csv")
print("\n>> Update path di hyperparameter_tuning.ipynb & phase1_screening.ipynb:")
print(f'   df_merged = joblib.load("{merged_path}")')


Saved:
  saved_models/df_merged_20260507_1049.joblib
  saved_models/adf_level_20260507_1049.csv
  saved_models/adf_transformed_20260507_1049.csv

>> Update path di hyperparameter_tuning.ipynb & phase1_screening.ipynb:
   df_merged = joblib.load("saved_models/df_merged_20260507_1049.joblib")


In [10]:
df_merged

,date,IHSG,BI_Rate,CPI,M2,NPL_Ratio,GDP,USDIDR,WTI,US_Treasury_10Y,Coal,Copper,Nickel,Silver,Tin,Gold,STI
0,2015-01-02,5242.769,7.75,80.29972,4174826.0,2.37,2.205429e+09,12542.5,52.69,2.114,61.30,6255.0,14830.0,15.7465,19645.0,1188.39,3370.59
1,2015-01-05,5219.995,7.75,80.29972,4174826.0,2.37,2.205429e+09,12627.5,50.04,2.034,61.25,6145.0,15200.0,16.1885,19495.0,1204.86,3328.28
2,2015-01-06,5169.060,7.75,80.29972,4174826.0,2.37,2.205429e+09,12657.5,47.93,1.938,61.45,6145.0,15260.0,16.5365,19775.0,1218.58,3281.95
3,2015-01-07,5207.118,7.75,80.29972,4174826.0,2.37,2.205429e+09,12738.5,48.65,1.969,60.60,6115.0,15550.0,16.5354,19700.0,1211.41,3298.36
4,2015-01-08,5211.828,7.75,80.29972,4174826.0,2.37,2.205429e+09,12680.0,48.79,2.018,60.60,6104.0,15550.0,16.3651,19780.0,1208.79,3345.11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2438,2025-01-22,7257.128,5.75,106.80000,9198352.0,2.18,3.329511e+09,16280.0,75.44,4.612,116.75,9223.5,15718.0,30.8283,30272.0,2756.48,3781.21
2439,2025-01-23,7232.643,5.75,106.80000,9198352.0,2.18,3.329511e+09,16275.0,74.62,4.645,116.50,9232.0,15668.0,30.4540,29899.0,2754.87,3806.57
2440,2025-01-24,7166.056,5.75,106.80000,9198352.0,2.18,3.329511e+09,16170.0,74.66,4.616,116.35,9276.0,15668.0,30.5850,30156.0,2770.58,3804.26
2441,2025-01-30,7073.478,5.75,106.80000,9198352.0,2.18,3.329511e+09,16255.0,72.73,4.519,114.85,9128.5,15394.0,31.5960,30269.0,2794.59,3804.26
